# Controlador Fuzzy de Apostas Esportivas - Mamdani com scikit-fuzzy

> **Disciplina:** Sistemas Baseados em Conhecimento (SBC)
> **Mini-Projeto 2 - Caminho B:** reescrita do dominio do Mini-Projeto 1 (recomendacao de apostas de futebol) trocando as regras IF-THEN nitidas por inferencia fuzzy.
> **Biblioteca:** scikit-fuzzy (controlador Mamdani, defuzzificacao por centroide)

---

## Visao geral da arquitetura

```
ENTRADAS (Antecedentes)                  INFERENCIA MAMDANI               SAIDA (Consequente)
-----------------------                  ------------------               -------------------
forma  (0-100)  ruim|media|boa  ---+
mando  (0-100)  fraco|medio|forte ---+--> 13 regras fuzzy  --> agregacao  --> confianca (0-100)
risco  (0-10)   baixo|medio|alto ---+    (min/AND, max/OR)   --> centroide     evitar|cautela|apostar
```

O controlador recebe tres variaveis continuas, aplica 13 regras fuzzy, agrega os
conjuntos de saida e **defuzzifica por centroide**, produzindo um numero continuo de
**confianca em apostar no mandante (0-100)**. Esse valor e mapeado para uma banda
qualitativa (BAIXA / MEDIA / ALTA), espelhando os limiares de score do MP1.

## Variaveis linguisticas e termos

| Variavel | Tipo | Universo | Termos |
|----------|------|----------|--------|
| `forma` | entrada | 0-100 | ruim, media, boa |
| `mando` | entrada | 0-100 | fraco, medio, forte |
| `risco` | entrada | 0-10 | baixo, medio, alto |
| `confianca` | saida | 0-100 | evitar, cautela, apostar |

## Base de regras (13)

| # | SE | ENTAO |
|---|----|-------|
| R1 | forma ruim E mando fraco | evitar |
| R2 | forma ruim E mando medio | evitar |
| R3 | forma ruim E mando forte | cautela |
| R4 | forma media E mando fraco | cautela |
| R5 | forma media E mando medio | cautela |
| R6 | forma media E mando forte | apostar |
| R7 | forma boa E mando fraco | cautela |
| R8 | forma boa E mando medio | apostar |
| R9 | forma boa E mando forte | apostar |
| R10 | risco alto | evitar |
| R11 | risco baixo E forma boa | apostar |
| R12 | risco medio E forma ruim | evitar |
| R13 | risco baixo E mando forte | apostar |

As regras R1-R9 formam a **grade completa `forma` x `mando`** (3x3 = 9 combinacoes):
como os termos cobrem todo o universo de cada variavel, qualquer entrada ativa ao menos
uma dessas regras, garantindo **ausencia de lacunas**. R10 e o override dominante de risco;
R11-R13 refinam a confianca conforme o nivel de risco.


## 0. Instalacao

Execute esta celula apenas se as bibliotecas ainda nao estiverem instaladas (necessario no Google Colab).

In [ ]:
!pip install scikit-fuzzy scipy networkx matplotlib -q

## 1. Imports

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

print('Bibliotecas importadas com sucesso')
print('scikit-fuzzy versao:', getattr(fuzz, '__version__', 'desconhecida'))

## 2. Variaveis linguisticas e funcoes de pertinencia

Cada variavel recebe no minimo 3 termos. As funcoes de pertinencia (MFs) sao
trapezoidais nos extremos (para saturar nos limites do universo) e triangulares no
meio. As sobreposicoes entre termos vizinhos sao intencionais: e nelas que o fuzzy
expressa a transicao gradual que o MP1 nao conseguia (cortes secos).

**Justificativa de dominio:**
- `forma` (0-100): indice de forma recente do mandante. No MP1 era discretizada por
  cortes secos (>=4 vitorias -> boa; >=3 derrotas -> ruim). Aqui e continua.
- `mando` (0-100): aproveitamento/forca de mando em casa. No MP1, `>= 70%` ativava
  `VantagemCasa` de forma abrupta. O fuzzy elimina o "penhasco" entre 69% e 70%.
- `risco` (0-10): risco contextual agregado (lesoes de titulares + clima + posicao na
  tabela). Condensa em um eixo continuo os fatos booleanos separados do MP1
  (LesoesCriticas, ClimaDesfavoravel, ClassificacaoRisco).
- `confianca` (0-100): saida unica e continua, substituindo o score 0-92 do MP1 que era
  calculado por tres formulas ad-hoc (modos positivo/empate/risco).

In [ ]:
# Universos de discurso
forma     = ctrl.Antecedent(np.arange(0, 101, 1),   'forma')
mando     = ctrl.Antecedent(np.arange(0, 101, 1),   'mando')
risco     = ctrl.Antecedent(np.arange(0, 10.1, 0.1),'risco')
confianca = ctrl.Consequent(np.arange(0, 101, 1),   'confianca')

# forma: forma recente do mandante (0 = pessima, 100 = excelente)
forma['ruim']  = fuzz.trapmf(forma.universe, [0, 0, 20, 40])
forma['media'] = fuzz.trimf (forma.universe, [30, 50, 70])
forma['boa']   = fuzz.trapmf(forma.universe, [60, 80, 100, 100])

# mando: aproveitamento / forca de mando em casa (%)
mando['fraco'] = fuzz.trapmf(mando.universe, [0, 0, 30, 50])
mando['medio'] = fuzz.trimf (mando.universe, [40, 60, 75])
mando['forte'] = fuzz.trapmf(mando.universe, [65, 80, 100, 100])

# risco: risco contextual agregado (lesoes + clima + tabela), 0 = nenhum, 10 = critico
risco['baixo'] = fuzz.trapmf(risco.universe, [0, 0, 2, 4])
risco['medio'] = fuzz.trimf (risco.universe, [3, 5, 7])
risco['alto']  = fuzz.trapmf(risco.universe, [6, 8, 10, 10])

# confianca: saida - confianca em apostar no mandante (0-100)
confianca['evitar']  = fuzz.trapmf(confianca.universe, [0, 0, 20, 40])
confianca['cautela'] = fuzz.trimf (confianca.universe, [30, 50, 70])
confianca['apostar'] = fuzz.trapmf(confianca.universe, [60, 80, 100, 100])

print('Variaveis e funcoes de pertinencia definidas')

### Visualizacao das funcoes de pertinencia

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 12))
for ax, var, titulo in zip(
        axes,
        [forma, mando, risco, confianca],
        ['forma (0-100)', 'mando (0-100)', 'risco (0-10)', 'confianca - SAIDA (0-100)']):
    for termo in var.terms:
        ax.plot(var.universe, var[termo].mf, linewidth=2, label=termo)
    ax.set_title(titulo)
    ax.set_ylabel('pertinencia')
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Base de regras fuzzy (13 regras)

As regras combinam os termos com `&` (E / operador min) e produzem um termo de saida.
R1-R9 cobrem toda a grade `forma` x `mando`; R10-R13 tratam o eixo `risco`.

In [ ]:
regras = [
    # --- Grade completa forma x mando (R1-R9): garante cobertura sem lacunas ---
    ctrl.Rule(forma['ruim']  & mando['fraco'], confianca['evitar'],  label='R1'),
    ctrl.Rule(forma['ruim']  & mando['medio'], confianca['evitar'],  label='R2'),
    ctrl.Rule(forma['ruim']  & mando['forte'], confianca['cautela'], label='R3'),
    ctrl.Rule(forma['media'] & mando['fraco'], confianca['cautela'], label='R4'),
    ctrl.Rule(forma['media'] & mando['medio'], confianca['cautela'], label='R5'),
    ctrl.Rule(forma['media'] & mando['forte'], confianca['apostar'], label='R6'),
    ctrl.Rule(forma['boa']   & mando['fraco'], confianca['cautela'], label='R7'),
    ctrl.Rule(forma['boa']   & mando['medio'], confianca['apostar'], label='R8'),
    ctrl.Rule(forma['boa']   & mando['forte'], confianca['apostar'], label='R9'),
    # --- Override e refinamento por risco (R10-R13) ---
    ctrl.Rule(risco['alto'],                    confianca['evitar'],  label='R10'),
    ctrl.Rule(risco['baixo'] & forma['boa'],    confianca['apostar'], label='R11'),
    ctrl.Rule(risco['medio'] & forma['ruim'],   confianca['evitar'],  label='R12'),
    ctrl.Rule(risco['baixo'] & mando['forte'],  confianca['apostar'], label='R13'),
]

print(f'Base com {len(regras)} regras definida')

## 4. Sistema de controle Mamdani

Montamos o `ControlSystem` com as 13 regras. A funcao `diagnostico` cria uma simulacao,
aplica as entradas, computa a inferencia, **defuzzifica por centroide** e imprime:
os graus de pertinencia ativados em cada entrada, o valor continuo de saida, a banda
qualitativa e uma interpretacao.

In [ ]:
sistema_ctrl = ctrl.ControlSystem(regras)


def banda(valor):
    '''Mapeia a confianca continua para uma banda qualitativa (limiares do MP1).'''
    if valor >= 65:
        return 'ALTA'
    if valor >= 40:
        return 'MEDIA'
    return 'BAIXA'


def recomendacao(valor):
    '''Traduz a banda em acao recomendada.'''
    return {'ALTA': 'APOSTAR no mandante',
            'MEDIA': 'CAUTELA / aposta de menor valor',
            'BAIXA': 'EVITAR a aposta'}[banda(valor)]


def graus(antecedente, valor):
    '''Graus de pertinencia de um valor crisp em cada termo do antecedente.'''
    return {t: round(float(fuzz.interp_membership(antecedente.universe,
                                                  antecedente[t].mf, valor)), 2)
            for t in antecedente.terms}


def diagnostico(titulo, forma_v, mando_v, risco_v, plot=True):
    '''Executa o controlador para um caso e exibe a explicacao completa.'''
    sim = ctrl.ControlSystemSimulation(sistema_ctrl)
    sim.input['forma'] = forma_v
    sim.input['mando'] = mando_v
    sim.input['risco'] = risco_v
    sim.compute()
    saida = sim.output['confianca']

    print('=' * 68)
    print(f'  {titulo}')
    print('=' * 68)
    print(f'  Entradas : forma={forma_v}  mando={mando_v}  risco={risco_v}')
    print(f'  forma -> {graus(forma, forma_v)}')
    print(f'  mando -> {graus(mando, mando_v)}')
    print(f'  risco -> {graus(risco, risco_v)}')
    print('  ' + '-' * 64)
    print(f'  Confianca (defuzzificada por centroide): {saida:.2f} / 100')
    print(f'  Banda: {banda(saida)}   ->   Recomendacao: {recomendacao(saida)}')
    print('=' * 68)
    if plot:
        confianca.view(sim=sim)
        plt.show()
    return saida


print('Sistema de controle e funcoes auxiliares prontos')

## 5. Casos de teste

Tres casos principais (espelhando os cenarios do MP1) mais um caso limitrofe que
evidencia a transicao suave do fuzzy. Cada caso mostra entradas, saida defuzzificada e
interpretacao comentada.

### Caso 1 - Favorito jogando em casa (esperado: ALTA / apostar)

Mandante em otima fase, forte aproveitamento em casa e sem riscos relevantes.
No MP1 este cenario resultava em `VITORIA_CASA`. Aqui esperamos confianca alta.
- `forma = 85` (boa), `mando = 80` (forte), `risco = 1` (baixo)

In [ ]:
# Caso 1: forma boa + mando forte + risco baixo -> confianca ALTA.
# Interpretacao: R9 (boa & forte -> apostar) e R11/R13 reforcam; saida ~84 = ALTA.
c1 = diagnostico('Caso 1 - Favorito em casa', forma_v=85, mando_v=80, risco_v=1)

### Caso 2 - Time em crise e com desfalques (esperado: BAIXA / evitar)

Mandante em pessima forma e com risco critico (lesoes + tabela). No MP1 o
`salience=90` da regra `EVITAR` dominava. Aqui o override R10 (risco alto -> evitar)
puxa a saida para baixo.
- `forma = 15` (ruim), `mando = 55` (medio), `risco = 9` (alto)

In [ ]:
# Caso 2: forma ruim + risco alto -> confianca BAIXA.
# Interpretacao: R10 (risco alto -> evitar) e R1/R2/R12 dominam; saida ~16 = BAIXA.
c2 = diagnostico('Caso 2 - Time em crise', forma_v=15, mando_v=55, risco_v=9)

### Caso 3 - Jogo equilibrado (esperado: MEDIA / cautela)

Times medianos, sem favoritismo claro e risco moderado. No MP1 caia na faixa de
`EMPATE`/confianca media. Aqui esperamos a saida ancorada em torno de 50.
- `forma = 50` (media), `mando = 55` (medio), `risco = 5` (medio)

In [ ]:
# Caso 3: tudo medio -> confianca MEDIA.
# Interpretacao: R5 (media & medio -> cautela) domina; saida ~50 = MEDIA.
c3 = diagnostico('Caso 3 - Jogo equilibrado', forma_v=50, mando_v=55, risco_v=5)

### Caso 4 - Limite de mando: a transicao suave (fuzzy x corte seco do MP1)

No MP1, `VantagemCasa` so era declarada com aproveitamento `>= 70%`: 69% e 70% davam
decisoes diferentes (um penhasco). Variando `mando` de 60 a 80 (com `forma=50` e
`risco=4`), o controlador fuzzy responde de forma **continua**, sem saltos.

In [ ]:
print('Sweep de mando (forma=50, risco=4): saida continua, sem penhasco em 70')
print('-' * 56)
valores_mando = [60, 65, 68, 69, 70, 71, 72, 75, 80]
saidas = []
for m in valores_mando:
    sim = ctrl.ControlSystemSimulation(sistema_ctrl)
    sim.input['forma'] = 50
    sim.input['mando'] = m
    sim.input['risco'] = 4
    sim.compute()
    v = sim.output['confianca']
    saidas.append(v)
    print(f'  mando={m:3d}  ->  confianca={v:6.2f}  [{banda(v)}]')

plt.figure(figsize=(8, 4))
plt.plot(valores_mando, saidas, 'o-', linewidth=2)
plt.axvline(70, color='red', linestyle='--', alpha=0.7,
            label='corte seco do MP1 (>=70 -> VantagemCasa)')
plt.title('Confianca x mando - resposta continua do fuzzy')
plt.xlabel('mando (aproveitamento em casa)')
plt.ylabel('confianca defuzzificada')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 6. Superficie de controle

Mapa da saida `confianca` em funcao de `forma` x `mando` (com `risco = 2` fixo). E o
analogo fuzzy da "analise de sensibilidade" em matriz do MP1, mas continuo: em vez de
celulas discretas com rotulos, temos um gradiente suave.

In [ ]:
passo = 4
fx = np.arange(0, 101, passo)
mx = np.arange(0, 101, passo)
Z = np.zeros((len(fx), len(mx)))
for i, fv in enumerate(fx):
    for j, mv in enumerate(mx):
        sim = ctrl.ControlSystemSimulation(sistema_ctrl)
        sim.input['forma'] = fv
        sim.input['mando'] = mv
        sim.input['risco'] = 2
        sim.compute()
        Z[i, j] = sim.output['confianca']

plt.figure(figsize=(8, 6))
cf = plt.contourf(mx, fx, Z, levels=20, cmap='RdYlGn')
plt.colorbar(cf, label='confianca (0-100)')
plt.title('Superficie de controle: confianca = f(forma, mando) | risco=2')
plt.xlabel('mando')
plt.ylabel('forma')
plt.show()
print('Saida minima:', round(Z.min(), 1), '| maxima:', round(Z.max(), 1))

## 7. Comparativo explicito: MP1 (regras nitidas) x MP2 (fuzzy)

### Complexidade

| Aspecto | MP1 - regras nitidas (`experta`) | MP2 - fuzzy (`scikit-fuzzy`) |
|---------|----------------------------------|------------------------------|
| Numero de regras | 14 | 13 |
| Estrutura | 3 niveis de encadeamento (fatos brutos -> derivados -> decisao) | 1 nivel (entradas -> saida) |
| Resolucao de conflito | `salience` manual (90 > 80 > 75 ...) | automatica (agregacao max + centroide) |
| Calculo da saida | 3 formulas ad-hoc de score (modos positivo/empate/risco) | defuzzificacao unica por centroide |
| Fatos intermediarios | 8 (VantagemCasa, LesoesCriticas, MomentumPositivo, ...) | nenhum |
| Esforco de manutencao | alto (ordenar salience, evitar conflito de regras) | baixo (adicionar/ajustar MFs e regras) |

### Expressividade

| Situacao | MP1 - regras nitidas | MP2 - fuzzy |
|----------|----------------------|-------------|
| Aproveitamento 69% vs 70% | decisoes diferentes (penhasco em >=70) | transicao continua e suave |
| "Forma quase boa" | inexprimivel (boolean: boa ou nao) | grau de pertinencia parcial |
| Combinacao de evidencias fracas | precisa de regra explicita | emerge da agregacao das MFs |
| Saida | rotulo discreto + score por formula | numero continuo + banda derivada |

### Sintese

- **Complexidade:** o MP1 precisa de encadeamento em niveis, fatos derivados, `salience`
  para ordenar disparos e formulas separadas de score. O MP2 elimina tudo isso: a
  matematica fuzzy (min/max + centroide) resolve conflitos e produz a saida sozinha.
- **Expressividade:** o MP1 raciocina em degraus (um valor pertence ou nao a uma
  categoria), o que cria penhascos artificiais em limiares como 70%. O MP2 representa
  gradacao e incerteza nativamente, gerando uma superficie de decisao continua.
- **Trade-off:** o MP1 oferece rastreabilidade passo a passo (a cadeia de regras
  disparadas e o porque de cada ponto de score) e decisoes categoricas diretas. O MP2
  oferece robustez a variacoes pequenas e suavidade, ao custo de uma explicabilidade
  menos literal (a saida vem da area agregada, nao de uma trilha de regras).


---
*Execute o notebook de cima para baixo. Testado em Python 3.11+ e Google Colab com scikit-fuzzy 0.5.0.*